In [0]:
data = [
    (1,'John',35),
    (2,"Rishab",32),
    (3,"Srini",47),
    (4,"Sally",51),
    
]
bronze_df = spark.createDataFrame(data,["emp_id","name","age"])

In [0]:
spark.sql("""
          CREATE TABLE if not exists employee_bronze (
            emp_id in,
            name string,
            age long
          )
          USING DELTA
          TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
          """)

In [0]:
%sql
INSERT INTO employee_bronze (emp_id, name, age)
VALUES
  (1, 'John', 30),
  (2, 'Mary', 40),
  (3, 'Mike', 50),
  (4, 'Jane', 60)

In [0]:
%sql
 DESCRIBE history employee_bronze

In [0]:
%sql
update employee_bronze set age = 35 where emp_id = 2;

In [0]:
%sql
insert into employee_bronze values (5, 'Bob', 25);

In [0]:
%sql
delete from employee_bronze where emp_id = 3;

In [0]:
%sql
DESCRIBE history employee_bronze;

In [0]:
cdf_df = spark.read \
    .format("delta") \
    .option("readChangeFeed", "true") \
    .option("startingVersion", 1) \
    .table("employee_bronze")

In [0]:
cdf_df.show()

In [0]:
final_silver = cdf_df.filter("""
                             _change_type in ("update_postimage","insert","delete")
                             """).orderBy("_commit_version","emp_id")

In [0]:
final_silver.show(truncate=False)